# Version 1

- This colorful html output worked well in terminal but failed in jupyter notebook

The reason your elegant logging engine works perfectly in a standard terminal but generates an empty HTML file in a Jupyter Notebook comes down to how Jupyter manages standard output ($sys.stdout$) and how CrewAI initializes its background workers.

Here is the breakdown of why this happens and how you can fix it.

---

## 🔍 Why it Fails in Jupyter Notebooks

### 1. Eager Stream Binding (The Main Culprit)

In your script, you instantiate your `Agent`, `Task`, and `Crew` **outside** of the `with capture_and_log(...)` context manager:

```python
tester_agent = Agent(...)
task1 = Task(...)
crew = Crew(agents=[tester_agent], tasks=[task1], verbose=True) # <-- Happens HERE

if __name__ == "__main__":
    with capture_and_log(TRANSCRIPT_FILE, HTML_FILE):
        crew.kickoff()

```

When `Crew(...)` is initialized, CrewAI and its underlying framework (LangChain) set up their internal logging and verbose printing systems.

* **In a Terminal:** Python resolves `sys.stdout` dynamically at runtime when a print statement occurs.
* **In Jupyter:** Jupyter replaces `sys.stdout` with a special object (`ipykernel.iostream.OutStream`) to route prints to your browser. When CrewAI initializes *outside* your context manager, it eagerly captures a direct reference to Jupyter's active `OutStream` object. When you later swap `sys.stdout` inside the `with` block, CrewAI completely ignores it and keeps writing directly to the cached Jupyter stream. Thus, your `PipelineLogger.write()` is never called.

### 2. Jupyter's Asynchronous Event Loop

Jupyter Notebooks run on a persistent, asynchronous event loop (`asyncio`). CrewAI also heavily relies on asynchronous execution to manage agents. In Jupyter, when async tasks yield or run in background threads, the IPython kernel explicitly forces output back to its own stream handlers, bypassing manual `sys.stdout` monkeypatching.

---

## 🛠️ The Fixes

To fix this, you need to ensure that CrewAI hooks into your `PipelineLogger` instead of Jupyter's default stream.

### Solution 1: Move Initialization Inside the Context Manager (Recommended)

The simplest and cleanest fix is to move your Agent, Task, and Crew setup **inside** the `with capture_and_log` block. This forces CrewAI to discover and bind to your `PipelineLogger` instead of the notebook's default stream.

Modify your execution block like this:

```python
# ====================== RUN ======================
if __name__ == "__main__":
    
    # Everything, including Agent/Crew setup, must happen INSIDE the manager
    with capture_and_log(TRANSCRIPT_FILE, HTML_FILE):
        
        logger.info("Main", "Setting up Agent inside the logging context...")
        tester_agent = Agent(
            role="System Tester",
            goal="Use the Dummy_Search tool to verify the system works.",
            backstory="Testing terminal colors in Jupyter.",
            tools=[dummy_search],
            llm=llm,
            verbose=True, 
            allow_delegation=False
        )

        task1 = Task(
            description="Call the Dummy_Search tool. Format final answer in Chinese and English.",
            expected_output="A short bilingual success message.",
            agent=tester_agent
        )

        crew = Crew(agents=[tester_agent], tasks=[task1], verbose=True)
        logger.info("Main", "✅ Crew successfully assembled within context.")
        
        user_request = "Run a quick ANSI color and HTML pipe test."
        logger.info("Main", f"🚀 Kickstarting Crew process with input: '{user_request}'")
        
        try:
            result = crew.kickoff(inputs={"user_request": user_request})
            logger.info("Main", "✅ Crew kickoff completed successfully.")
            print(f"\n{'='*50}\nFINAL RESULT\n{'='*50}\n{result}")
            
        except Exception as e:
            logger.error("Main", f"❌ Fatal error during Crew execution: {e}", exc_info=True)

```

### Solution 2: Force Global Python Logging to Intercept

Because CrewAI utilizes standard Python logging under the hood, if Solution 1 still leaves gaps due to Jupyter's async behavior, you can explicitly attach a `StreamHandler` pointing to your interceptor inside your `PipelineLogger`.

Add this method to your `PipelineLogger` class:

```python
class PipelineLogger:
    def __init__(self, text_path, html_path):
        # ... your existing init code ...
        
        # Force Python's root logger to also stream to this interceptor
        import logging
        self.logging_handler = logging.StreamHandler(self)
        logging.getLogger().addHandler(self.logging_handler)

    # ... keep your existing write, flush, etc. ...

    def generate_html_and_close(self):
        # Remove the logging handler when closing to prevent memory leaks
        import logging
        logging.getLogger().removeHandler(self.logging_handler)
        
        # ... your existing HTML generation code ...

```

### Summary of Best Practices for Jupyter

* Run the entire block of code in **a single Jupyter notebook cell**.
* Keep any library that uses heavy background logging/printing (like CrewAI) instantiated *inside* your custom stdout override wrapper.

In [1]:
import os
import sys
import datetime
# Ensure current directory is in path
sys.path.append(os.getcwd())
from config.api_key import API_KEY_MOONSHOT
from crewai import Agent, Task, Crew, LLM
from crewai.tools import tool
from utils.htmlcolorlog import capture_and_log
from utils.logger import setup_logger

In [2]:
# ====================== CONFIG ======================
BASE_DIR = "D:/AI_Data/Computer/WorldQuantBrain-Agent/"
CHROMA_DIR = BASE_DIR + "embedding_db/quant_forum_chroma/"
BGEM3_DIR = BASE_DIR + "embedding_db/quant_forum_bgem3/"
HF_CACHE_DIR = BASE_DIR + "cache/hf/"
PIP_CACHE_DIR = BASE_DIR + "cache/pip/"
LOG_DIR = BASE_DIR + "logs/" + datetime.datetime.now().strftime("%Y%m") + "/"

for directory in [CHROMA_DIR, HF_CACHE_DIR, PIP_CACHE_DIR, BGEM3_DIR, LOG_DIR]:
    os.makedirs(directory, exist_ok=True)

# Define file paths for our transcript and HTML logs
timestamp = datetime.datetime.now().strftime('%Y%m%d-%H%M%S')
TRANSCRIPT_FILE = os.path.join(LOG_DIR, f"wqb_agent-{timestamp}.transcript.txt")
HTML_FILE = os.path.join(LOG_DIR, f"wqb_agent-{timestamp}.html")

logger = setup_logger(LOG_DIR, "wqb_agent_test", "wqb_main_logger")

# ====================== YOUR GEMINI CLIENT ======================
logger.info("Main", "Initializing LLM client...")
llm = LLM(
    model="moonshot/moonshot-v1-32k",   
    base_url="https://api.moonshot.cn/v1",
    api_key=API_KEY_MOONSHOT,
    temperature=0.6,          
    max_tokens=8192,
    timeout=180,              
    max_retries=5,            
)
logger.info("Main", "✅ LLM client initialized successfully.")

# -------------------------------------------------------------------------
# 🛑 SKIPPED FOR TESTING: HuggingFace Embeddings, PyPDFLoader, and ChromaDB 
# This prevents the script from hanging for a long time during the test.
# -------------------------------------------------------------------------

# ====================== DUMMY SEARCH TOOL ======================
@tool("Dummy_Search")
def dummy_search(query: str) -> str:
    """A fake search tool to test if the agent can use tools and log output.
    IMPORTANT FORMATTING RULE: 
    The 'query' argument MUST be a plain string. 
    DO NOT pass a dictionary.
    For example, 
    Correct: "momentum" 
    Incorrect: {"type": "str", "value": "momentum"}
    """
    logger.info("Dummy Search", f"🔍 Tool Called: 'Dummy_Search' | Query: '{query}'")
    return "This is dummy data. Tell the user the test is successful."

[26-5-20 09:48:57][INFO][SETUP LOG] ✅ logger System Started
[26-5-20 09:48:57][INFO][SETUP LOG] Log file path: D:/AI_Data/Computer/WorldQuantBrain-Agent/logs/202605/wqb_agent_test-20260520-094857.log
[26-5-20 09:48:57][INFO][Main] Initializing LLM client...
[26-5-20 09:48:57][INFO][Main] ✅ LLM client initialized successfully.


In [3]:
# ====================== AGENT (Simplified) ======================
logger.info("Main", "Setting up Agent...")
tester_agent = Agent(
    role="System Tester",
    goal="Use the Dummy_Search tool to verify the system works, then output a short success message.",
    backstory="You are a quick test agent checking if terminal colors and emojis pipe correctly to HTML.",
    tools=[dummy_search],
    llm=llm,
    verbose=True, # <-- This ensures CrewAI prints the colorful output
    allow_delegation=False
)

# ====================== TASK & CREW (Simplified) ======================
logger.info("Main", "Defining Tasks and assembling Crew...")
task1 = Task(
    description="Call the Dummy_Search tool with the query 'test formatting'. Then format your final answer in Chinese and English.",
    expected_output="A short bilingual success message.",
    agent=tester_agent
)

crew = Crew(agents=[tester_agent], tasks=[task1], verbose=True)
logger.info("Main", "✅ Crew successfully assembled.")

# ====================== RUN ======================
if __name__ == "__main__":
    # ELEGANT PART: Everything inside this block is captured, formatted, and exported safely.
    # The with capture_and_log(...) block is the magic here. Even if your CrewAI code crashes 
    # in the middle of execution, Python's Context Manager guarantees that sys.stdout will be 
    # restored back to normal and the HTML file will be generated properly.
    with capture_and_log(TRANSCRIPT_FILE, HTML_FILE):
        user_request = "Run a quick ANSI color and HTML pipe test."
        logger.info("Main", f"🚀 Kickstarting Crew process with input: '{user_request}'")
        
        try:
            result = crew.kickoff(inputs={"user_request": user_request})
            logger.info("Main", "✅ Crew kickoff completed successfully.")
            print(f"\n{'='*50}\nFINAL RESULT\n{'='*50}\n{result}")
            
        except Exception as e:
            logger.error("Main", f"❌ Fatal error during Crew execution: {e}", exc_info=True)

[26-5-20 09:49:01][INFO][Main] Setting up Agent...
[26-5-20 09:49:01][INFO][Main] Defining Tasks and assembling Crew...
[26-5-20 09:49:01][INFO][Main] ✅ Crew successfully assembled.
[26-5-20 09:49:01][INFO][Main] 🚀 Kickstarting Crew process with input: 'Run a quick ANSI color and HTML pipe test.'


🚀 Kickstarting Crew process with input: 'Run a quick ANSI color and HTML pipe test.'


╭──────────────────────────────────────────── Crew Execution Started ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: dd382730-c83e-40cc-8f17-148dddbb43e7                                                                       │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: System Tester                                                                                           │
│                                                                                                                 │
│  Task: Call the Dummy_Search tool with the query 'test formatting'. Then format your final answer in Chinese    │
│  and English.                                                                                                   │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Output()

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: System Tester                                                                                           │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  成功消息：终端颜色和emoji已经正确地输出到HTML。Success message: Terminal colors and emojis are outputting      │
│  correctly to HTML.                                                                                             │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Task Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: 78d5d0b7-b912-4d19-8a75-56962ba73870                                                                     │
│  Agent: System Tester                                                                                           │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: dd382730-c83e-40cc-8f17-148dddbb43e7                                                                       │
│  Tool Args:                                                                                                     │
│  Final Output: 成功消息：终端颜色和emoji已经正确地输出到HTML。Success message: Terminal colors and emojis are   │
│  outputting correctly to HTML.                                                                                  │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[26-5-20 09:49:06][INFO][Main] ✅ Crew kickoff completed successfully.


✅ Crew kickoff completed successfully.

FINAL RESULT
成功消息：终端颜色和emoji已经正确地输出到HTML。Success message: Terminal colors and emojis are outputting correctly to HTML.

[INFO] 🌐 Colorful HTML Log saved to: D:/AI_Data/Computer/WorldQuantBrain-Agent/logs/202605/wqb_agent-20260520-094857.html


# Version 2

In [4]:
# ====================== RUN ======================
# if __name__ == "__main__":

# ELEGANT PART: Everything inside this block is captured, formatted, and exported safely.
# The with capture_and_log(...) block is the magic here. Even if your CrewAI code crashes 
# in the middle of execution, Python's Context Manager guarantees that sys.stdout will be 
# restored back to normal and the HTML file will be generated properly.
with capture_and_log(TRANSCRIPT_FILE, HTML_FILE):
    # ====================== AGENT (Simplified) ======================
    logger.info("Main", "Setting up Agent...")
    tester_agent = Agent(
        role="System Tester",
        goal="Use the Dummy_Search tool to verify the system works, then output a short success message.",
        backstory="You are a quick test agent checking if terminal colors and emojis pipe correctly to HTML.",
        tools=[dummy_search],
        llm=llm,
        verbose=True, # <-- This ensures CrewAI prints the colorful output
        allow_delegation=False
    )

    # ====================== TASK & CREW (Simplified) ======================
    logger.info("Main", "Defining Tasks and assembling Crew...")
    task1 = Task(
        description="Call the Dummy_Search tool with the query 'test formatting'. Then format your final answer in Chinese and English.",
        expected_output="A short bilingual success message.",
        agent=tester_agent
    )

    crew = Crew(agents=[tester_agent], tasks=[task1], verbose=True)
    logger.info("Main", "✅ Crew successfully assembled.")
    user_request = "Run a quick ANSI color and HTML pipe test."
    logger.info("Main", f"🚀 Kickstarting Crew process with input: '{user_request}'")
    
    try:
        result = crew.kickoff(inputs={"user_request": user_request})
        logger.info("Main", "✅ Crew kickoff completed successfully.")
        print(f"\n{'='*50}\nFINAL RESULT\n{'='*50}\n{result}")
        
    except Exception as e:
        logger.error("Main", f"❌ Fatal error during Crew execution: {e}", exc_info=True)

[26-5-20 09:45:37][INFO][Main] Setting up Agent...
[26-5-20 09:45:37][INFO][Main] Defining Tasks and assembling Crew...
[26-5-20 09:45:37][INFO][Main] ✅ Crew successfully assembled.
[26-5-20 09:45:37][INFO][Main] 🚀 Kickstarting Crew process with input: 'Run a quick ANSI color and HTML pipe test.'


╭──────────────────────────────────────────── Crew Execution Started ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 6012b2e8-a053-46b6-859f-333e13c76215                                                                       │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: System Tester                                                                                           │
│                                                                                                                 │
│  Task: Call the Dummy_Search tool with the query 'test formatting'. Then format your final answer in Chinese    │
│  and English.                                                                                                   │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Output()

[26-5-20 09:45:40][INFO][Dummy Search] 🔍 Tool Called: 'Dummy_Search' | Query: 'test formatting'


...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: System Tester                                                                                           │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  `系统测试成功：终端颜色和表情符号已正确地管道到HTML。` / `System test successful: Terminal colors and emojis   │
│  have been piped correctly to HTML.`                                                                            │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Task Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: 78365e3a-63cf-4783-b2ab-f3a5ab223d0b                                                                     │
│  Agent: System Tester                                                                                           │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 6012b2e8-a053-46b6-859f-333e13c76215                                                                       │
│  Tool Args:                                                                                                     │
│  Final Output: `系统测试成功：终端颜色和表情符号已正确地管道到HTML。` / `System test successful: Terminal       │
│  colors and emojis have been piped correctly to HTML.`                                                          │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[26-5-20 09:45:42][INFO][Main] ✅ Crew kickoff completed successfully.



FINAL RESULT
`系统测试成功：终端颜色和表情符号已正确地管道到HTML。` / `System test successful: Terminal colors and emojis have been piped correctly to HTML.`

[INFO] 🌐 Colorful HTML Log saved to: D:/AI_Data/Computer/WorldQuantBrain-Agent/logs/202605/wqb_agent-20260520-094528.html
